# Pandas Mastery for Data Science
Comprehensive guide to pandas operations.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
%matplotlib inline
pd.set_option('display.max_columns', 20)

## 1. DataFrame Creation and I/O

In [ ]:
# Various ways to create DataFrames
df1 = pd.DataFrame({'name': ['Alice', 'Bob', 'Charlie'], 'age': [25, 30, 35], 'salary': [50000, 60000, 70000]})
print("From dict:\n", df1)

df2 = pd.DataFrame(np.random.randn(5, 3), columns=['A', 'B', 'C'], index=pd.date_range('2024-01-01', periods=5))
print("\nFrom numpy with DatetimeIndex:\n", df2)

# From list of dicts
records = [{'x': 1, 'y': 2}, {'x': 3, 'y': 4, 'z': 5}]
df3 = pd.DataFrame(records)
print("\nFrom records (note NaN):\n", df3)

## 2. Indexing: loc, iloc, Boolean Masking

In [ ]:
np.random.seed(42)
df = pd.DataFrame(np.random.randn(6, 4), columns=['A','B','C','D'],
                  index=['r1','r2','r3','r4','r5','r6'])
print("DataFrame:\n", df.round(2))

print("\n--- loc (label-based) ---")
print("df.loc['r2', 'B']:", df.loc['r2', 'B'].round(2))
print("df.loc['r1':'r3', ['A','C']]:\n", df.loc['r1':'r3', ['A','C']].round(2))

print("\n--- iloc (position-based) ---")
print("df.iloc[0, 1]:", df.iloc[0, 1].round(2))
print("df.iloc[0:2, 0:2]:\n", df.iloc[0:2, 0:2].round(2))

print("\n--- Boolean masking ---")
mask = df['A'] > 0
print(f"Rows where A > 0:\n", df[mask].round(2))

## 3. GroupBy Operations

In [ ]:
# Create realistic dataset
np.random.seed(42)
n = 200
sales = pd.DataFrame({
    'store': np.random.choice(['NYC', 'LA', 'Chicago', 'Houston'], n),
    'product': np.random.choice(['Electronics', 'Clothing', 'Food'], n),
    'revenue': np.random.exponential(100, n),
    'quantity': np.random.randint(1, 20, n),
    'date': pd.date_range('2024-01-01', periods=n, freq='D')
})

print("=== GroupBy: Single Column ===")
print(sales.groupby('store')['revenue'].agg(['mean', 'sum', 'count']).round(2))

print("\n=== GroupBy: Multiple Columns ===")
print(sales.groupby(['store', 'product'])['revenue'].mean().unstack().round(2))

print("\n=== Named Aggregations ===")
result = sales.groupby('store').agg(
    total_revenue=('revenue', 'sum'),
    avg_quantity=('quantity', 'mean'),
    num_transactions=('revenue', 'count')
).round(2)
print(result)

## 4. Merge/Join Patterns

In [ ]:
# Create example tables
customers = pd.DataFrame({'id': [1,2,3,4], 'name': ['Alice','Bob','Charlie','Diana']})
orders = pd.DataFrame({'order_id': [101,102,103,104,105],
                       'customer_id': [1,2,2,5,1],
                       'amount': [50, 30, 80, 20, 60]})

print("Customers:\n", customers)
print("\nOrders:\n", orders)

print("\n--- INNER JOIN (only matching) ---")
print(pd.merge(customers, orders, left_on='id', right_on='customer_id', how='inner'))

print("\n--- LEFT JOIN (all customers) ---")
print(pd.merge(customers, orders, left_on='id', right_on='customer_id', how='left'))

print("\n--- OUTER JOIN (all records) ---")
print(pd.merge(customers, orders, left_on='id', right_on='customer_id', how='outer'))

## 5. Window Functions

In [ ]:
# Time series with window functions
np.random.seed(42)
ts = pd.DataFrame({
    'date': pd.date_range('2024-01-01', periods=100, freq='D'),
    'value': np.cumsum(np.random.randn(100)) + 50
}).set_index('date')

ts['rolling_7'] = ts['value'].rolling(7).mean()
ts['rolling_21'] = ts['value'].rolling(21).mean()
ts['expanding_mean'] = ts['value'].expanding().mean()
ts['ewm'] = ts['value'].ewm(span=10).mean()

fig, ax = plt.subplots(figsize=(12, 5))
ts[['value', 'rolling_7', 'rolling_21', 'ewm']].plot(ax=ax, lw=[0.8, 2, 2, 2])
ax.set_title('Window Functions: Rolling, Expanding, EWM')
ax.set_ylabel('Value')
plt.legend()
plt.tight_layout()
plt.show()

## 6. Method Chaining

In [ ]:
# Clean, readable data transformations using method chaining
result = (sales
    .assign(unit_price=lambda x: x['revenue'] / x['quantity'])
    .query('revenue > 50')
    .groupby('store')
    .agg(avg_unit_price=('unit_price', 'mean'),
         total_revenue=('revenue', 'sum'))
    .sort_values('total_revenue', ascending=False)
    .round(2)
)
print("Method chaining result:\n", result)

# Compare with verbose style
print("\n(Same result, but method chaining is much more readable!)")

## 7. Performance: apply vs Vectorized

In [ ]:
n = 100000
df_perf = pd.DataFrame({'a': np.random.randn(n), 'b': np.random.randn(n)})

# Method 1: apply (slow)
start = time.perf_counter()
result1 = df_perf.apply(lambda row: row['a']**2 + row['b']**2, axis=1)
t_apply = time.perf_counter() - start

# Method 2: vectorized (fast)
start = time.perf_counter()
result2 = df_perf['a']**2 + df_perf['b']**2
t_vec = time.perf_counter() - start

# Method 3: numpy (fastest)
start = time.perf_counter()
result3 = df_perf['a'].values**2 + df_perf['b'].values**2
t_np = time.perf_counter() - start

print(f"apply (row-wise): {t_apply*1000:.1f} ms")
print(f"pandas vectorized: {t_vec*1000:.1f} ms  ({t_apply/t_vec:.0f}x faster)")
print(f"numpy vectorized:  {t_np*1000:.1f} ms  ({t_apply/t_np:.0f}x faster)")

# Verify same results
assert np.allclose(result1.values, result2.values)
print("\nAll methods produce identical results ✓")

## 8. Time Series Functionality

In [ ]:
# Resample, shift, diff
dates = pd.date_range('2023-01-01', '2024-12-31', freq='D')
ts_data = pd.DataFrame({
    'date': dates,
    'sales': np.random.poisson(50, len(dates)) + np.sin(np.arange(len(dates))*2*np.pi/365)*20
}).set_index('date')

print("=== Resample (daily -> monthly) ===")
monthly = ts_data.resample('M').agg(['sum', 'mean'])
print(monthly.head())

print("\n=== Shift (lag features) ===")
ts_data['lag_1'] = ts_data['sales'].shift(1)
ts_data['lag_7'] = ts_data['sales'].shift(7)
ts_data['diff'] = ts_data['sales'].diff()
ts_data['pct_change'] = ts_data['sales'].pct_change()
print(ts_data.head(10))

## 9. Pivot Tables and Cross-Tabs

In [ ]:
# Pivot table
pivot = sales.pivot_table(
    values='revenue', 
    index='store', 
    columns='product', 
    aggfunc=['mean', 'count']
).round(2)
print("Pivot Table:\n", pivot)

# Cross-tab
print("\nCross-Tab (counts):")
print(pd.crosstab(sales['store'], sales['product']))

# Visualization
sales.pivot_table(values='revenue', index='store', columns='product', aggfunc='mean').plot(
    kind='bar', figsize=(8, 4), title='Average Revenue by Store and Product')
plt.ylabel('Revenue')
plt.tight_layout()
plt.show()

## 10. Handling Missing Data

In [ ]:
# Create data with missing values
df_missing = pd.DataFrame({
    'A': [1, np.nan, 3, np.nan, 5],
    'B': [np.nan, 2, np.nan, 4, 5],
    'C': [1, 2, 3, 4, 5],
    'D': [np.nan, np.nan, np.nan, 4, 5]
})
print("Original:\n", df_missing)
print(f"\nMissing counts:\n{df_missing.isnull().sum()}")
print(f"\nMissing percentage:\n{(df_missing.isnull().sum()/len(df_missing)*100).round(1)}%")

print("\n--- Strategies ---")
print("Drop rows with any NaN:\n", df_missing.dropna())
print("\nFill with mean:\n", df_missing.fillna(df_missing.mean()).round(2))
print("\nForward fill:\n", df_missing.ffill())
print("\nInterpolate:\n", df_missing.interpolate())

## Key Pandas Tips

1. **Avoid loops** - use vectorized operations and groupby
2. **Use `.values` or `.to_numpy()`** when you need raw performance
3. **Method chaining** makes code readable and debuggable
4. **Categorical dtype** saves memory for repeated strings
5. **Set proper index** for time series operations